In [1]:
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import tiktoken
from dataclasses import dataclass
from huggingface_hub import hf_hub_download, login
from safetensors.torch import load_file


REPO_ID = "hiianish/mini-gpt2_decoder"


@dataclass
class GPTConfig:
    vocab_size: int = 50257
    context_length: int = 256
    n_layers: int = 6
    n_heads: int = 6
    n_embd: int = 384
    dropout: float = 0.1
    bias: bool = True
    use_fused_kernel: bool = False

    def __post_init__(self):
        assert self.n_embd % self.n_heads == 0


class LayerNorm(nn.Module):
    def __init__(self, ndim, bias=True):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None

    def forward(self, x):
        return F.layer_norm(
            x,
            self.weight.shape,
            self.weight,
            self.bias,
            1e-5
        )


class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.n_heads = config.n_heads
        self.n_embd = config.n_embd
        self.head_dim = config.n_embd // config.n_heads

        self.qkv_proj = nn.Linear(
            config.n_embd,
            3 * config.n_embd,
            bias=config.bias
        )

        self.out_proj = nn.Linear(
            config.n_embd,
            config.n_embd,
            bias=config.bias
        )

        self.resid_dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        B, T, C = x.size()

        qkv = self.qkv_proj(x)

        q, k, v = qkv.split(self.n_embd, dim=2)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        y = F.scaled_dot_product_attention(
            q,
            k,
            v,
            is_causal=True
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)

        y = self.out_proj(y)
        y = self.resid_dropout(y)

        return y


class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.fc_in = nn.Linear(
            config.n_embd,
            4 * config.n_embd,
            bias=config.bias
        )

        self.gelu = nn.GELU()

        self.fc_out = nn.Linear(
            4 * config.n_embd,
            config.n_embd,
            bias=config.bias
        )

        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x):
        x = self.fc_in(x)
        x = self.gelu(x)
        x = self.fc_out(x)
        x = self.dropout(x)
        return x


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.ln_1 = LayerNorm(
            config.n_embd,
            bias=config.bias
        )

        self.attn = CausalSelfAttention(config)

        self.ln_2 = LayerNorm(
            config.n_embd,
            bias=config.bias
        )

        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


class MiniGPT(nn.Module):
    def __init__(self, config):
        super().__init__()

        self.config = config

        self.token_embedding = nn.Embedding(
            config.vocab_size,
            config.n_embd
        )

        self.position_embedding = nn.Embedding(
            config.context_length,
            config.n_embd
        )

        self.dropout = nn.Dropout(config.dropout)

        self.blocks = nn.ModuleList([
            Block(config)
            for _ in range(config.n_layers)
        ])

        self.ln_f = LayerNorm(
            config.n_embd,
            bias=config.bias
        )

        self.lm_head = nn.Linear(
            config.n_embd,
            config.vocab_size,
            bias=False
        )

        self.token_embedding.weight = self.lm_head.weight

    def forward(self, idx):
        B, T = idx.shape

        if T > self.config.context_length:
            idx = idx[:, -self.config.context_length:]
            T = idx.shape[1]

        positions = torch.arange(
            0,
            T,
            dtype=torch.long,
            device=idx.device
        )

        tok_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(positions)

        x = self.dropout(tok_emb + pos_emb)

        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)

        logits = self.lm_head(x)

        return logits

    @torch.no_grad()
    def generate(
        self,
        idx,
        max_new_tokens,
        temperature=1.0,
        top_k=None
    ):
        self.eval()

        for _ in range(max_new_tokens):

            idx_cond = (
                idx
                if idx.size(1) <= self.config.context_length
                else idx[:, -self.config.context_length:]
            )

            logits = self(idx_cond)

            logits = logits[:, -1, :]

            logits = logits / temperature

            if top_k is not None:
                v, _ = torch.topk(
                    logits,
                    min(top_k, logits.size(-1))
                )

                logits[logits < v[:, [-1]]] = -float("inf")

            probs = F.softmax(logits, dim=-1)

            idx_next = torch.multinomial(
                probs,
                num_samples=1
            )

            idx = torch.cat(
                (idx, idx_next),
                dim=1
            )

        return idx


def load_model():

    print("Downloading model files...")

    config_path = hf_hub_download(
        repo_id=REPO_ID,
        filename="config.json"
    )

    model_path = hf_hub_download(
        repo_id=REPO_ID,
        filename="model.safetensors"
    )

    with open(config_path, "r") as f:
        config_dict = json.load(f)

    allowed_keys = {
        "vocab_size",
        "context_length",
        "n_layers",
        "n_heads",
        "n_embd",
        "dropout",
        "bias",
        "use_fused_kernel"
    }

    config_dict = {
        k: v
        for k, v in config_dict.items()
        if k in allowed_keys
    }

    config = GPTConfig(**config_dict)

    print("Model configuration:")
    print(config)

    model = MiniGPT(config)

    state_dict = load_file(model_path)

    print(f"Checkpoint contains {len(state_dict)} tensors.")

    new_state_dict = {}

    for key, value in state_dict.items():

        new_key = key

        prefixes = [
            "transformer.",
            "model.",
            "module."
        ]

        for prefix in prefixes:
            if new_key.startswith(prefix):
                new_key = new_key[len(prefix):]

        new_state_dict[new_key] = value

    state_dict = new_state_dict

    model_state = model.state_dict()

    filtered_state_dict = {}

    for key, value in state_dict.items():

        if key in model_state:
            if model_state[key].shape == value.shape:
                filtered_state_dict[key] = value

    missing = [
        key
        for key in model_state.keys()
        if key not in filtered_state_dict
    ]

    unexpected = [
        key
        for key in state_dict.keys()
        if key not in model_state
    ]

    if missing:
        print("\nMissing keys:")
        for key in missing:
            print(" ", key)

    if unexpected:
        print("\nUnexpected keys:")
        for key in unexpected:
            print(" ", key)

    model.load_state_dict(
        filtered_state_dict,
        strict=False
    )

    device = (
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )

    model = model.to(device)
    model.eval()

    print(f"\nModel loaded successfully on {device}")

    return model, device


def generate_text(
    model,
    device,
    tokenizer,
    prompt,
    max_new_tokens=100,
    temperature=0.8,
    top_k=50
):

    tokens = tokenizer.encode_ordinary(prompt)

    input_ids = torch.tensor(
        [tokens],
        dtype=torch.long,
        device=device
    )

    output_ids = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        top_k=top_k
    )

    output_tokens = output_ids[0].tolist()

    return tokenizer.decode(output_tokens)


def main():

    model, device = load_model()

    tokenizer = tiktoken.get_encoding("gpt2")

    print("\nMiniGPT is ready.")
    print("Type 'exit' to quit.\n")

    while True:

        prompt = input("Prompt: ")

        if prompt.lower() == "exit":
            break

        if not prompt.strip():
            continue

        output = generate_text(
            model=model,
            device=device,
            tokenizer=tokenizer,
            prompt=prompt,
            max_new_tokens=100,
            temperature=0.8,
            top_k=50
        )

        print("\nGenerated:")
        print(output)
        print()


if __name__ == "__main__":
    main()

config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  120MB            

model.safetensors: downloading bytes:           |  0.00B            

Model configuration:
GPTConfig(vocab_size=50257, context_length=256, n_layers=6, n_heads=6, n_embd=384, dropout=0.1, bias=True, use_fused_kernel=False)
Checkpoint contains 76 tensors.

Missing keys:
  token_embedding.weight

Model loaded successfully on cuda

MiniGPT is ready.
Type 'exit' to quit.


Generated:
hii, and you want to work up, but you want to go back to you on the right way you know? And if you’ll see a lot of time, you’ll find some different things to think from the way you know, in the end of the game and the game.

If you’re going to play around, you really can’t get some big team from. You’re using a lot more of things that’s a


Generated:
what is 2+2.0.1+1 x1+5 +2=2+2+5 +2+5+0 +100 x1.6 -2+4" x3^2 +1 +5.0.0 +2 +2.1+1+0.3=6.7% +2.1.1.0.1 1.0x0.3.0% +0.3 .0.1.9 +3.2.

